>1: 1: Imports and Config

In [10]:
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 200)

DATA_DIR = "data/"
OUTPUT_DIR = "outputs/"

#For online data there is no column "Bill_Class"

SALES_DATA_PATH = os.path.join(DATA_DIR, "/Users/anandguntuku/Movies/VSCodeFolder/SarvaniMLOps/Data/dfx1_of_jul21_.csv")
HOLIDAY_DATA_PATH = os.path.join(DATA_DIR, "/Users/anandguntuku/Movies/VSCodeFolder/SarvaniMLOps/Data/Holiday_Data copy.csv")
DATE_TABLE_PATH = os.path.join(DATA_DIR, "/Users/anandguntuku/Movies/VSCodeFolder/SarvaniMLOps/Data/date_time_table.csv")
MERGED_FILE_PATH = os.path.join(OUTPUT_DIR, "/Users/anandguntuku/Movies/VSCodeFolder/SarvaniMLOps/Data/merged_df.csv")


>2: SweetShopDataManager (Master Class)

In [ ]:
class SweetShopDataManager:
    def __init__(self,
                 sales_path=SALES_DATA_PATH,
                 holiday_path=HOLIDAY_DATA_PATH,
                 date_path=DATE_TABLE_PATH,
                 merged_path=MERGED_FILE_PATH,
                 auto_load=True):
        
        self.sales_path = sales_path
        self.holiday_path = holiday_path
        self.date_path = date_path
        self.merged_path = merged_path
        self.df = None
        
        if auto_load:
            self.df = self._prepare_data()
    
    def _load_sales(self):
        usecols = ['Outlet', 'Date_Time', 'Bill_No', 'Corrected_Item_Name', 
                   'Portion_Code', 'Corrected_Item_Category', 'Item_Price', 
                   'Item_Quantity', 'Item_Total_Amount', 'Bill_Item_Count', 
                   'Order_Type', 'Weekday', 'Total_Bill_Amount_Corrected', 
                   'Daily_Bill_Amount', 'total_weight_kg', 'Festivity', 'Bill_Class']
        
        df = pd.read_csv(self.sales_path, usecols=usecols, parse_dates=['Date_Time'])
        df['Date'] = df['Date_Time'].dt.date.astype(str)
        return df

    def _load_holiday(self):
        df = pd.read_csv(self.holiday_path)
        df['Date'] = pd.to_datetime(df['Date']).dt.date.astype(str)
        return df

    def _load_date_table(self):
        df = pd.read_csv(self.date_path, parse_dates=['Date_Time'])
        df['Date'] = df['Date_Time'].dt.date.astype(str)
        return df

    def _prepare_data(self):
        if os.path.exists(self.merged_path):
            print("📂 Loading cached merged file...")
            return pd.read_csv(self.merged_path, parse_dates=['Date_Time'])
        
        print("🔄 Preparing merged data...")
        sales = self._load_sales()
        holiday = self._load_holiday()
        date = self._load_date_table()

        merged = pd.merge(
            sales,
            date,
            on=['Bill_No', 'Date_Time', 'Date'],
            how='left'
        )

        merged = pd.merge(
            merged,
            holiday,
            on='Date',
            how='left'
        )

        merged.rename(columns={'Holiday': 'Holiday_Name'}, inplace=True)

        os.makedirs(os.path.dirname(self.merged_path), exist_ok=True)
        merged.to_csv(self.merged_path, index=False)
        print(f"✅ Merged data saved to: {self.merged_path}")
        return merged

    def get_data(self):
        return self.df


>3: Use It Anywhere

In [9]:
# Instantiate and access data in one line
manager = SweetShopDataManager()
merged_df = manager.get_data()

# Preview
print(f"Shape: {merged_df.shape}")
merged_df.head()


🔄 Preparing merged data...
✅ Merged data saved to: /Users/anandguntuku/Movies/VSCodeFolder/SarvaniMLOps/Data/merged_df.csv
Shape: (6407267, 29)


,Outlet,Date_Time,Bill_No,Item_Price,Item_Quantity,Item_Total_Amount,Bill_Item_Count,Order_Type,Weekday,Total_Bill_Amount_Corrected,Daily_Bill_Amount,Corrected_Item_Name,Portion_Code,Corrected_Item_Category,total_weight_kg,Festivity_x,Bill_Class,Date,Unnamed: 0,Year,Month,Week_Number,Day,Festivity_y,Quarter,Festivity_Group,Day_Type,DayFestGroup,Holiday_Name
0,MADDILAPALEM OUTLET,2022-04-01 06:54:17.779,2022211,266.6667,1.0,279.9967,1,general,4,280.0,209884.0,KOVABISCUITS(ORANGE)0500G,0500G,KOVA SWEETS0500G,0.50,regular,Small,2022-04-01,33,2022,4,13,1,regular,Q1,Non-Festive,Wknd,Wknd - Non-Festive,NaN
1,MADDILAPALEM OUTLET,2022-04-01 08:27:52.593,2022212,180.9524,1.0,190.0024,1,general,4,190.0,209884.0,MOTHICHOORLADDURED0500G,0500G,LADDUS0500G,0.50,regular,Micro,2022-04-01,34,2022,4,13,1,regular,Q1,Non-Festive,Wknd,Wknd - Non-Festive,NaN
2,MADDILAPALEM OUTLET,2022-04-01 08:56:43.153,2022213,128.5714,2.0,270.0029,2,general,4,385.0,209884.0,BELLAMPUTHAREKULU0250G,0250G,HOME FOODS0250G,0.50,regular,Small,2022-04-01,35,2022,4,13,1,regular,Q1,Non-Festive,Wknd,Wknd - Non-Festive,NaN
3,MADDILAPALEM OUTLET,2022-04-01 08:56:43.158,2022213,109.5238,1.0,115.0038,2,general,4,385.0,209884.0,CHAMCHAM0250G,0250G,BENGALI SYRUP ITEMS0250G,0.25,regular,Small,2022-04-01,36,2022,4,13,1,regular,Q1,Non-Festive,Wknd,Wknd - Non-Festive,NaN
4,MADDILAPALEM OUTLET,2022-04-01 11:01:31.613,2022218,90.4762,1.0,94.9962,3,general,4,285.0,209884.0,BOBBATLU0250G,0250G,HOME FOODS0250G,0.25,regular,Small,2022-04-01,44,2022,4,13,1,regular,Q1,Non-Festive,Wknd,Wknd - Non-Festive,NaN


In [18]:
merged_df.sample(n=20)

,Outlet,Date_Time,Bill_No,Item_Price,Item_Quantity,Item_Total_Amount,Bill_Item_Count,Order_Type,Weekday,Total_Bill_Amount_Corrected,Daily_Bill_Amount,Corrected_Item_Name,Portion_Code,Corrected_Item_Category,total_weight_kg,Festivity_x,Bill_Class,Date,Unnamed: 0,Year,Month,Week_Number,Day,Festivity_y,Quarter,Festivity_Group,Day_Type,DayFestGroup,Holiday_Name
547629,MALKAPURAM OUTLET,2022-07-28 16:08:17.598,20221557894,57.1429,2.0,119.9957,3,general,3,342.0,545582.0,CHILLYCHICKENROLL01PI,01PI,PUFFS & ROLLS01PI,NaN,regular,Small,2022-07-28,878707,2022,7,30,28,regular,Q2,Non-Festive,Wkdy,Wkdy - Non-Festive,NaN
1407476,MALKAPURAM OUTLET,2024-11-19 11:30:08.401,202415107612,70.0000,1.0,70.0000,1,general,1,70.0,326678.0,PREMIUMBADAMMILK(BOTTLE)01B,01B,JUICES01B,NaN,regular,Micro,2024-11-19,1944709,2024,11,47,19,regular,Q3,Non-Festive,Wkdy,Wkdy - Non-Festive,NaN
6185543,EENADU OUTLET,2024-08-24 19:05:20.704,20241343953,30.4762,1.0,31.9962,2,general,5,67.0,336616.0,CHANDRAKALA0050G,0050G,MAIDHA SWEETS0050G,0.05,regular,Micro,2024-08-24,8298775,2024,8,34,24,regular,Q2,Non-Festive,Wknd,Wknd - Non-Festive,NaN
4509552,ASILMETTA OUTLET,2022-12-21 21:52:27.987,20221281179,133.3333,1.0,140.0033,4,general,2,540.0,288243.0,VEG-FRIED RICE1PLATE,1PLATE,RICE1PLATE,NaN,regular,Small,2022-12-21,6077619,2022,12,51,21,regular,Q3,Non-Festive,Wkdy,Wkdy - Non-Festive,NaN
936879,MALKAPURAM OUTLET,2024-10-20 14:45:39.029,20241595054,704.7619,1.0,740.0019,2,general,6,1190.0,584752.0,GHEEMIXEDSWEETS1000G,1000G,HOME FOODS1000G,1.00,regular,Medium,2024-10-20,1274522,2024,10,42,20,regular,Q3,Non-Festive,Wknd,Wknd - Non-Festive,NaN
4573680,ASILMETTA OUTLET,2023-07-31 08:44:07.638,20231224102,200.0000,1.0,210.0000,2,general,0,480.0,243095.0,BOBBATLU0500G,0500G,HOME FOODS0500G,0.50,regular,Small,2023-07-31,6142880,2023,7,31,31,regular,Q2,Non-Festive,Wkdy,Wkdy - Non-Festive,NaN
3812783,SIRIPURAM,2023-12-04 20:05:10.706,20231741116,49.1071,1.0,54.9971,5,general,0,335.0,222007.0,ALOOMASALACHIPS0100G,0100G,HOTS0100G,0.10,regular,Small,2023-12-04,5169661,2023,12,49,4,regular,Q3,Non-Festive,Wkdy,Wkdy - Non-Festive,NaN
5167170,WHEATRICH,2024-06-01 21:07:36.492,20241817164,50.0000,3.0,150.0000,1,general,5,150.0,140592.0,PREMIUMBADAMMILK(BOTTLE)01B,01B,JUICES01B,NaN,regular,Micro,2024-06-01,6844847,2024,6,22,1,regular,Q1,Non-Festive,Wknd,Wknd - Non-Festive,NaN
2495463,CHINAMUSRIWADA OUTLET,2023-05-30 12:17:55.292,2023227511,28.5714,1.0,30.0014,3,general,1,165.0,153341.0,VEGPUFF01PI,01PI,PUFFS & ROLLS01PI,NaN,regular,Micro,2023-05-30,3334613,2023,5,22,30,regular,Q1,Non-Festive,Wkdy,Wkdy - Non-Festive,NaN
1141805,MALKAPURAM OUTLET,2022-12-08 18:33:32.332,202215130437,703.3898,1.0,829.9998,2,general,3,860.0,595139.0,BUTTERSCOOTCHEGGLESS01U,01U,PASTRIES01U,NaN,regular,Medium,2022-12-08,1674822,2022,12,49,8,regular,Q3,Non-Festive,Wkdy,Wkdy - Non-Festive,NaN
